# `every_eval_ever` → INIF workflow

This notebook pulls an instance-level config from the [`evaleval/EEE_datastore`](https://huggingface.co/datasets/evaleval/EEE_datastore) collection on HuggingFace, converts it into an `InifDocument` with real tokenization, and demonstrates the full INIF analysis loop.

[`every_eval_ever`](https://github.com/evaleval/every_eval_ever) is a crowdsourced schema-standardized database of LLM evaluation runs — contributions come from Inspect AI, HELM, lm-eval-harness, and more. Each entry conforms to the `instance_level_eval_0.2.2` schema (input / output / messages / scores / token usage). INIF consumes that schema directly through `inif.converters.evaleval`.

Chosen config: **`theory_of_mind_samples`** — a manageably sized Theory-of-Mind benchmark where the evaluated model is `Qwen/Qwen2.5-3B-Instruct` (whose tokenizer we can load locally).

For multi-turn or agentic traces, swap the config for e.g. `wordle_arena_samples` — the converter handles `single_turn`, `multi_turn`, and `agentic` interaction types uniformly.

In [10]:
%%capture
!uv pip install "inif[evaleval]" transformers

## 1. Load instance records from EEE_datastore

`from_hf_dataset` streams rows from the HuggingFace dataset (no full download), tokenizes each record with the model's own tokenizer via `apply_chat_template`, runs sequence deduplication, and tags chat roles. `limit=30` keeps this tutorial fast; omit it to process the whole config.

The `tokenizer` argument defaults to `"auto"`: the converter reads the model id off each record (after stripping any routing prefix like `together/` or `hf/`) and calls `AutoTokenizer.from_pretrained` for you. Pass an explicit string or tokenizer instance to override; if it disagrees with the source's `model_id` you'll get a `UserWarning` (the mismatch may be intentional, e.g. running one model's outputs through a different model's tokenizer for cross-model studies).

In [11]:
import warnings

warnings.filterwarnings("ignore")

from inif.converters.evaleval import from_hf_dataset  # noqa: E402

# tokenizer="auto" (the default) → load the tokenizer matching record["model_id"]
doc = from_hf_dataset("theory_of_mind_samples", limit=30)

meta = doc.metadata
src = meta.source_eval
print(f"Model:           {meta.model.name}")
print(f"Framework:       {src.framework} v{src.framework_version}")
print(f"Task:            {src.task}")
print(f"Samples:         {doc.total_samples}")
print(f"Sequences:       {len(doc.sequences)} (shared token runs)")
print(f"Source:          {meta.sources[-1] if meta.sources else '(none)'}")

Model:           Qwen/Qwen2.5-3B-Instruct
Framework:       evaleval v0.2.2
Task:            theory_of_mind
Samples:         30
Sequences:       0 (shared token runs)
Source:          hf://evaleval/EEE_datastore/theory_of_mind_samples[samples]


## 2. Inspect the converted structure

Each `Sample` carries the tokenized chat stream (with dedup'd sequence refs), the scorer result, the extracted answer, and the token-usage counts that the EEE schema records per sample. Per-token roles are stored in the `role` extra and per-token tags include `generated` (assistant response) and, where applicable, `reasoning`.

In [12]:
from collections import Counter
from statistics import mean, median

interaction_counts = Counter(s.metadata.get("interaction_type") for s in doc.samples)
print("Interaction types:", dict(interaction_counts))

inp = [s.input_tokens for s in doc.samples if s.input_tokens is not None]
out = [s.output_tokens for s in doc.samples if s.output_tokens is not None]
print("\nToken usage (from EEE):")
if inp:
    print(
        f"  input:   mean={mean(inp):6.1f}  median={median(inp):6.0f}  max={max(inp)}"
    )
if out:
    print(
        f"  output:  mean={mean(out):6.1f}  median={median(out):6.0f}  max={max(out)}"
    )

s0 = doc.samples[0]
n_ref = sum(1 for t in s0.tokens if t.is_sequence_ref)
n_own = sum(1 for t in s0.tokens if not t.is_sequence_ref)
print(f"\nSample '{s0.id}':")
print(f"  target:     {s0.target!r}")
print(
    f"  score:      {s0.scores[0].value}  (answer={s0.scores[0].answer!r}, "
    f"is_correct={s0.scores[0].metadata.get('is_correct')})"
)
print(f"  tokens:     {n_ref} sequence ref(s) + {n_own} own tokens")

role_counts = Counter(
    t.get_extra("role", "?") for t in s0.tokens if not t.is_sequence_ref
)
print(f"  roles:      {dict(role_counts)}")

Interaction types: {'single_turn': 30}

Token usage (from EEE):
  input:   mean= 102.0  median=   102  max=102
  output:  mean= 272.0  median=   272  max=272

Sample '1':
  target:     'bathtub'
  score:      1.0  (answer=None, is_correct=True)
  tokens:     0 sequence ref(s) + 0 own tokens
  roles:      {}


## 3. Enrich with content-level tags

The converter already adds `role` extras and `generated` tags. On top of that, a couple of lines of regex tagging highlight answer-candidate tokens. `tag_by_text_regex_all` works on the concatenated token stream, so BPE-split words are handled natively.

In [13]:
from inif import tag_by_text_regex_all
from inif.tagging import TextTagMode

# Tag single-word candidate answers inside the assistant response (role=assistant).
tag_by_text_regex_all(
    doc,
    pattern=r"\b[A-Za-z_]{3,20}\b",
    tag="word",
    mode=TextTagMode.FIRST,
)

first = doc.samples[0]
tagged_words = [
    t.token
    for t in first.tokens
    if t.has_tag("word") and t.get_extra("role") == "assistant"
][:15]
print(f"First 15 'word' tags inside the assistant response of sample {first.id!r}:")
print("  ", tagged_words)

First 15 'word' tags inside the assistant response of sample '1':
   []


## 4. Correct vs incorrect: does the model ramble more when it's wrong?

`filter_samples_by_score` partitions the document by the per-instance `evaluation.score` carried through the EEE schema. Comparing mean output-token counts across correct and incorrect samples is a common first pass on a new benchmark.

In [14]:
from statistics import mean

from inif import filter_samples_by_score

correct = filter_samples_by_score(doc, "theory_of_mind", lambda v: v >= 0.5)
incorrect = filter_samples_by_score(doc, "theory_of_mind", lambda v: v < 0.5)


def assistant_token_count(sample) -> int:
    return sum(
        1
        for t in sample.tokens
        if not t.is_sequence_ref and t.get_extra("role") == "assistant"
    )


def mean_or_nan(xs):
    return mean(xs) if xs else float("nan")


c_out = [s.output_tokens for s in correct if s.output_tokens is not None]
i_out = [s.output_tokens for s in incorrect if s.output_tokens is not None]
c_asst = [assistant_token_count(s) for s in correct]
i_asst = [assistant_token_count(s) for s in incorrect]

print(
    f"Correct   (n={len(correct):2d})  "
    f"output_tokens mean={mean_or_nan(c_out):6.1f}  "
    f"assistant tokens (ours) mean={mean_or_nan(c_asst):6.1f}"
)
print(
    f"Incorrect (n={len(incorrect):2d})  "
    f"output_tokens mean={mean_or_nan(i_out):6.1f}  "
    f"assistant tokens (ours) mean={mean_or_nan(i_asst):6.1f}"
)

Correct   (n=26)  output_tokens mean= 272.0  assistant tokens (ours) mean=   0.0
Incorrect (n= 4)  output_tokens mean= 272.0  assistant tokens (ours) mean=   0.0


## 5. Visualise a single trace

`show` renders the document inline as colored HTML. We pull one correct and one incorrect trace into self-contained sub-documents with `doc.subset` so the renderer only draws those.

In [15]:
from inif import show

if incorrect:
    target_id = incorrect[0].id
    print(f"Showing the first INCORRECT sample: {target_id}")
    show(doc.subset(lambda s: s.id == target_id))
else:
    target_id = doc.samples[0].id
    print(f"No incorrect samples in this slice — showing first sample: {target_id}")
    show(doc.subset(lambda s: s.id == target_id))

Showing the first INCORRECT sample: 5


## 6. Persist and validate

Save the enriched document and round-trip it through the INIF schema validator to confirm the EEE-sourced data is wire-compatible with the rest of the INIF ecosystem.

In [16]:
from inif import load, save, to_dict, validate

save(doc, "theory_of_mind.inif.json")
save(doc, "theory_of_mind.inif")  # gzipped

validate(to_dict(doc))

reloaded = load("theory_of_mind.inif")
print(
    f"Round-tripped: {reloaded.total_samples} samples, "
    f"{len(reloaded.sequences)} sequences, "
    f"framework={reloaded.metadata.source_eval.framework}"
)

Round-tripped: 30 samples, 0 sequences, framework=evaleval


In [17]:
from inif import viewer

viewer.show(doc)

## Scaling note

Full EEE configs can run into 10–100k records. For those, use `save_shards` / `iter_shards` to split the document and process shard-by-shard — expansion and heavy enrichment then fit in memory even for the largest benchmarks. For aggregate regex / statistical scans, `FlatTokenStore.from_document` gives a single flat view over all expanded tokens without materializing them per-sample.

In [18]:
from inif import FlatTokenStore, iter_shards, save_shards

save_shards(doc, "tom_shards", samples_per_shard=10, compress=True)

for shard in iter_shards("tom_shards"):
    store = FlatTokenStore.from_document(shard)
    print(
        f"shard: {len(shard.samples)} samples, {store.total_tokens:,} expanded tokens"
    )
    break

shard: 10 samples, 0 expanded tokens
